In [305]:
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # repo root
sys.path.insert(0, str(ROOT))
##
import pandas as pd 
import numpy as np
import math
from src.features.feature_engineering import add_time_and_lag_features
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from sklearn.base import clone
from sklearn.linear_model import Ridge
from gurobipy import Model, GRB, quicksum
from k_means_constrained import KMeansConstrained
import folium
import webbrowser
import time

### parameters 

In [306]:
hour_to_optimize = '2017-11-14 08:00:00'
total_nb_trucks = 10

### Loading processed data

In [307]:
# if this cell is not working please run the models.ipynb notebook to generate the processed data csv
demand = pd.read_csv("../data/processed/divvy_hourly_demand_weather.csv")
demand["hour"] = pd.to_datetime(demand["hour"])
demand = add_time_and_lag_features(demand=demand,lags=(1, 24, 168),drop_na_lags=True)
demand.head()

,station_id,hour,departures,arrivals,net_demand,temp,tmin,tmax,rhum,prcp,...,dow_sin,dow_cos,month_sin,month_cos,dep_lag_1,arr_lag_1,dep_lag_24,arr_lag_24,dep_lag_168,arr_lag_168
168,2,2017-01-08 00:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,1.0,0.0,0.0,0.0,0.0
169,2,2017-01-08 01:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
170,2,2017-01-08 02:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
171,2,2017-01-08 03:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
172,2,2017-01-08 04:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0


In [308]:
stations = pd.read_csv("../data/raw/Divvy_Stations_2017_Q3Q4 (2).csv")
stations.drop(columns=["city","online_date","Unnamed: 7"], inplace=True)

warehouse_row = pd.DataFrame({
    "id" : [0],
    "name" : ["Divvy_Warehouse"],
    # "latitude" : [41.9507379911931] ,
    # "longitude" : [-87.80493131681082],
    "latitude" : [41.88994543021417] ,
    "longitude" : [-87.6804617],   
    "dpcapacity": [0]
})
stations = pd.concat([stations,warehouse_row], ignore_index=True)

### Demand prediction

In [309]:
y_dep = demand["departures"]
y_arr = demand["arrivals"]

feature_cols = [
    "hour_sin","hour_cos",
    "dow_sin","dow_cos",
    "month_sin","month_cos",
    "is_weekend",
    "temp","tmin","tmax","rhum","prcp","snwd","wspd","pres",
    "dep_lag_1","dep_lag_24", 'dep_lag_168',
    "arr_lag_1","arr_lag_24", "arr_lag_168"
]

X = demand[feature_cols]

split_time = demand["hour"].quantile(0.8)

train = demand["hour"] <= split_time
test  = demand["hour"] > split_time

X_train = X[train]
X_test  = X[test]

y_dep_train = y_dep[train]
y_dep_test  = y_dep[test]

y_arr_train = y_arr[train]
y_arr_test  = y_arr[test]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [310]:
model = Ridge(alpha=1.0, random_state=42)

print(f"Training Ridge...")

# departures
mdl_dep = clone(model)
mdl_dep.fit(X_train, y_dep_train)
pred_dep = mdl_dep.predict(X_test)

# arrivals
mdl_arr = clone(model)
mdl_arr.fit(X_train, y_arr_train)
pred_arr = mdl_arr.predict(X_test)

pred_dep = np.clip(pred_dep, 0, None)
pred_arr = np.clip(pred_arr, 0, None)

pred_net = pred_dep - pred_arr
true_net = y_dep_test - y_arr_test

results = pd.DataFrame([{
    "model": "Ridge",
    "dep_MAE": mean_absolute_error(y_dep_test, pred_dep),
    "dep_RMSE": rmse(y_dep_test, pred_dep),
    "arr_MAE": mean_absolute_error(y_arr_test, pred_arr),
    "arr_RMSE": rmse(y_arr_test, pred_arr),
    "net_MAE": mean_absolute_error(true_net, pred_net),
    "net_RMSE": rmse(true_net, pred_net),
}])
results.head()

Training Ridge...


,model,dep_MAE,dep_RMSE,arr_MAE,arr_RMSE,net_MAE,net_RMSE
0,Ridge,0.374209,0.945263,0.373062,0.969262,0.456264,1.099768


In [311]:
# build pred_df 
pred_df = demand[test][['station_id','hour','departures','arrivals','net_demand']].rename(columns={
    "net_demand" : "true_net_demand",
    "departures" : "true_departures",
    "arrivals" : "true_arrivals"
})
pred_df['pred_arrivals'] = pred_arr.round()
pred_df['pred_departures'] = pred_dep.round()
pred_df['pred_net_demand'] = pred_net.round()

pred_df = pred_df.merge(stations, left_on='station_id', right_on='id', how='left')
pred_df.drop(columns=['id'], inplace=True)
pred_df = pred_df[['hour','station_id','pred_departures', 'true_departures','pred_arrivals', 'true_arrivals',
       'pred_net_demand','true_net_demand','dpcapacity', 'latitude', 'longitude','name']]

hourly_net_demand = pred_df.groupby(pred_df['hour'])[['pred_net_demand','true_net_demand']].agg(lambda x: x.abs().sum()).reset_index().sort_values(['pred_net_demand'], ascending=False)
hourly_net_demand[hourly_net_demand['hour']==hour_to_optimize]



,hour,pred_net_demand,true_net_demand
573,2017-11-14 08:00:00,762.0,1146


### data prep for optimization

In [312]:
hto = pred_df[pred_df['hour']==hour_to_optimize]
hto.describe()

,hour,station_id,pred_departures,true_departures,pred_arrivals,true_arrivals,pred_net_demand,true_net_demand,dpcapacity,latitude,longitude
count,585,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000
mean,2017-11-14 08:00:00,317.312821,1.688889,1.876923,1.675214,1.907692,0.013675,0.030769,17.463248,41.887417,-87.658172
min,2017-11-14 08:00:00,2.000000,0.000000,0.000000,0.000000,0.000000,-26.000000,-17.000000,0.000000,41.736646,-87.803911
25%,2017-11-14 08:00:00,163.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,15.000000,41.851375,-87.679459
50%,2017-11-14 08:00:00,315.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,15.000000,41.886923,-87.653833
75%,2017-11-14 08:00:00,475.000000,2.000000,2.000000,2.000000,2.000000,1.000000,0.000000,19.000000,41.931320,-87.630834
max,2017-11-14 08:00:00,626.000000,36.000000,41.000000,30.000000,49.000000,13.000000,36.000000,55.000000,42.064313,-87.549386
std,NaN,181.022135,3.132184,3.676363,3.649988,4.680368,2.944664,4.176632,6.747838,0.067122,0.043398


In [313]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    distance = R * c
    return distance

In [314]:
np.random.seed(8)
hto['current_bike_nb'] = (np.clip(
        np.random.normal(0.5 * hto['dpcapacity'], 0.2 * hto['dpcapacity']),
        0,
        hto['dpcapacity']
    )).astype(int)

# sanity check 

print(hto['current_bike_nb'].sum()/hto['dpcapacity'].sum())

0.47484338292873923


### Filtering layer

In [315]:
acceptable_margin = 0.2

target = 0.5 * hto['dpcapacity']
deviation = (hto['current_bike_nb'] - target).abs()

candidate_stations_ids = hto.loc[
    deviation >= acceptable_margin * hto['dpcapacity'],
    'station_id'
].tolist()
candidate_stations = stations[stations['id'].isin(candidate_stations_ids)]

### Geo clustering 

In [316]:
# Nb of clusters
n_clusters = int(np.floor(len(candidate_stations)/25))
size_min = 20
size_max = 30

# Coordinates matrix
coords = candidate_stations[['latitude', 'longitude']]

# Fit 
kmeans_constrained = KMeansConstrained(n_clusters=n_clusters, size_min=size_min, size_max=size_max, random_state=8, n_init=20)
candidate_stations['cluster'] = kmeans_constrained.fit_predict(coords)
cluster = candidate_stations.groupby('cluster')['id'].count().reset_index(name='stations_nb')
cluster['proportion'] = cluster['stations_nb'] / cluster['stations_nb'].sum()
cluster['assigned_trucks'] = np.round((cluster['proportion'] * total_nb_trucks)).astype(int)

In [317]:
cluster_balance_check = candidate_stations.merge(hto, left_on='id', right_on='station_id', how='left')
cluster_balance_check.groupby('cluster')['pred_net_demand'].sum()

cluster
0    21.0
1     1.0
2     4.0
3     7.0
4     0.0
5   -58.0
6     0.0
7    12.0
Name: pred_net_demand, dtype: float64

In [318]:
center_lat = candidate_stations['latitude'].mean()
center_lon = candidate_stations['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

colors = [
    'red', 'blue', 'green', 'purple', 'orange', 
    'yellow', 'pink', 'cyan', 'magenta', 'lime', 
    'teal', 'brown', 'navy', 'gold', 'violet', 
    'indigo', 'turquoise', 'coral', 'salmon', 'olive'
]

for _, row in candidate_stations.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color=colors[row['cluster']],
        fill=True,
        fill_color=colors[row['cluster']],
        fill_opacity=0.8,
        tooltip=f"Station {row['id']} | Cluster {row['cluster']}"
    ).add_to(m)

m.save("station_clusters.html")
#webbrowser.open("station_clusters.html")

### optimization

In [319]:
def bike_rebalancing(cluster_nb,
                     candidate_stations,
                     stations,
                     hto,
                     gamma,
                     nb_trucks):
    selected_station_ids = candidate_stations[candidate_stations['cluster']==cluster_nb]['id'].to_list()
    selected_station_ids = selected_station_ids + [0]
    w_stations = stations[stations['id'].isin(selected_station_ids)]
    w_hto = hto[hto['station_id'].isin(selected_station_ids)]

    # Sets
    I_all = w_stations['id'].to_list() # all stations id + warehouse 
    I = w_stations['id'][:-1].to_list() # all stations id without warehouse
    K = range(nb_trucks) # all trucks

    # Parameters
    alpha = {i: 0.2 for i in I} # service-level requirement
    C = {i: w_stations.loc[w_stations['id'] == i, 'dpcapacity'].values[0] for i in I} # capacity

    # S = {
    #     i: int(np.clip(
    #         np.random.normal(0.5 * C[i], 0.2 * C[i]),
    #         0,
    #         C[i]
    #     ))
    #     for i in I
    # }
    S = {i: w_hto.loc[w_hto['station_id'] == i, 'current_bike_nb'].values[0] for i in I}

    delta = {i: w_hto.loc[w_hto['station_id'] == i, 'pred_net_demand'].values[0] for i in I}  # predicted demand
    D = {i: {} for i in I_all}     # distance matrix D[i][j]
    for i in I_all :
        lat_i = w_stations.loc[w_stations['id'] == i, 'latitude'].values[0]
        lon_i = w_stations.loc[w_stations['id'] == i, 'longitude'].values[0]
        for j in I_all : 
            lat_j = w_stations.loc[w_stations['id'] == j, 'latitude'].values[0]
            lon_j = w_stations.loc[w_stations['id'] == j, 'longitude'].values[0]
            D[i][j] = haversine(lat_i, lon_i, lat_j, lon_j)
    M = 200
    Tmax = {k: 30 for k in K} # truck capacities
    warehouse_id = 0

    ######################
    # Model init
    model = Model("BikeRebalancing")

    # Decision Variables
    X = model.addVars(I, K, vtype=GRB.CONTINUOUS, name="X") # bikes drop off by truck k at station i
    Y = model.addVars(I, K, vtype=GRB.CONTINUOUS, name="Y") # bikes picked up by truck k at station i
    W = model.addVars(I_all, I_all, K, vtype=GRB.BINARY, name="W") # trip between ith and jth station
    T = model.addVars(I_all, K, vtype=GRB.CONTINUOUS, name="T") # nb of bikes before entering station i
    P = model.addVars(I, vtype=GRB.CONTINUOUS, name="P") # bike penalty associated with station i
    Q = model.addVars(I, vtype=GRB.CONTINUOUS, name="Q") # dock penalty associated with station i
    y = model.addVars(I, vtype=GRB.BINARY, name="y")
    z = model.addVars(I, vtype=GRB.BINARY, name="z")
    U = model.addVars(I_all, K, vtype=GRB.INTEGER, name="U") # visit order of truck k 

    ###################
    # objective 
    model.setObjective(
        quicksum(W[i,j,k]*D[i][j] for i in I_all for j in I_all for k in K) + 
        gamma * quicksum(P[i]+Q[i] for i in I),
        GRB.MINIMIZE
    )
    ###################
    # Constraints

    # Non-negativity (maybe not necessary)

    for i in I:
        for k in K:
            model.addConstr(X[i,k] >= 0)
            model.addConstr(Y[i,k] >= 0)

    for i in I_all:
        for k in K:
            model.addConstr(T[i,k] >= 0)

    # Visit order bounds

    for i in I:
        for k in K:
            model.addConstr(U[i,k] >= 1)
            model.addConstr(U[i,k] <= len(I))
    for k in K:
        model.addConstr(U[warehouse_id,k] == 0)


    # Stable system
    model.addConstr(quicksum(X[i,k] for i in I for k in K) == quicksum(Y[i,k] for i in I for k in K))

    # Station capacity
    for i in I:
        model.addConstr(S[i] + quicksum(X[i,k] for k in K) - quicksum(Y[i,k] for k in K) <= C[i])
        model.addConstr(S[i] + quicksum(X[i,k] for k in K) - quicksum(Y[i,k] for k in K) >= 0)

    # Truck routing logic

    for i in I:
        for k in K:
            model.addConstr(quicksum(W[i,j,k] for j in I_all) <= X[i,k] + Y[i,k])
            model.addConstr(X[i,k] + Y[i,k] <= M * quicksum(W[i,j,k] for j in I_all))

    for i in I_all:
        for k in K:
            model.addConstr(quicksum(W[i,j,k] for j in I_all) <= 1)
            model.addConstr(quicksum(W[i,j,k] for j in I_all) == quicksum(W[j,i,k] for j in I_all))
            model.addConstr(W[i,i,k] == 0)
    for k in K:
        model.addConstr(quicksum(W[warehouse_id,j,k] for j in I_all) == 1)

    # Truck bike-storing logic
    for i in I_all:
        for k in K:
            model.addConstr(T[i,k] <= Tmax[k])
            model.addConstr(T[i,k] <= M * (1 - W[warehouse_id,i,k]))

    for i in I : 
        for k in K:
            model.addConstr(X[i,k] <= T[i,k])
            model.addConstr(Y[i,k] <= Tmax[k] - T[i,k])

    for i in I:
        for j in I_all:
            if i != j:
                for k in K:
                    model.addConstr(T[j,k] <= T[i,k] + Y[i,k] - X[i,k] + M*(1 - W[i,j,k]))
                    model.addConstr(T[j,k] >= T[i,k] + Y[i,k] - X[i,k] - M*(1 - W[i,j,k]))
                
    for j in I_all:
        if 0 != j:
            for k in K:
                model.addConstr(T[j,k] <= T[0,k] + M*(1 - W[0,j,k]))
                model.addConstr(T[j,k] >= T[0,k] - M*(1 - W[0,j,k]))
    for k in K:
        model.addConstr(T[warehouse_id,k] == 0)

    # Service-level constraints (exclude depot)
    for i in I:
        model.addConstr(alpha[i]*C[i] - (S[i] - delta[i] + sum(X[i,k] for k in K) - sum(Y[i,k] for k in K)) <= P[i])
        model.addConstr(P[i] <= M * y[i])
        model.addConstr(alpha[i]*C[i] - (S[i] - delta[i] + sum(X[i,k] for k in K) - sum(Y[i,k] for k in K)) >= P[i] - M*(1-y[i]))
        model.addConstr((alpha[i]-1)*C[i] + (S[i] - delta[i] + sum(X[i,k] for k in K) - sum(Y[i,k] for k in K)) <= Q[i])
        model.addConstr(Q[i] <= M * z[i])
        model.addConstr((alpha[i]-1)*C[i] + (S[i] - delta[i] + sum(X[i,k] for k in K) - sum(Y[i,k] for k in K)) >= Q[i] - M*(1-z[i]))


    # MTZ subtour elimination (exclude depot)
    for i in I:
        for j in I:
            if i != j:
                for k in K:
                    model.addConstr(U[i,k] - U[j,k] + len(I) * W[i,j,k] <= len(I)-1)
                

    model.update()
    #model.printStats()
    model.setParam("OutputFlag", 0)
    model.Params.MIPGap = 0.05
    model.Params.TimeLimit = 60
    model.optimize()

# --- Gather solution ---
    solution = {
        "X": {(i,k): X[i,k].X for i in I for k in K},
        "Y": {(i,k): Y[i,k].X for i in I for k in K},
        "T": {(i,k): T[i,k].X for i in I_all for k in K},
        "W": {(i,j,k): W[i,j,k].X for i in I_all for j in I_all for k in K},
        "P": {i: P[i].X for i in I},
        "Q": {i: Q[i].X for i in I},
        "y": {i: y[i].X for i in I},
        "z": {i: z[i].X for i in I},
        "U": {(i,k): U[i,k].X for i in I_all for k in K},
        "objective": model.ObjVal,
        # --- Extra info for easy reporting ---
        "D": D,
        "I": I,
        "I_all": I_all,
        "K": list(K),
        "warehouse_id": warehouse_id,
        "S": S,
        "delta": delta,
        "C": C,
        "Tmax": Tmax
    }

    return solution

In [320]:
solutions_dict = {}
for cluster in range(n_clusters):
    print(f'Optimizing cluster {cluster}...')
    start_time = time.time()
    solutions_dict[cluster] = bike_rebalancing(cluster_nb=cluster,
                            candidate_stations=candidate_stations,
                            stations=stations,
                            hto=hto, 
                            gamma=10, #10000
                            nb_trucks=1)
    end_time = time.time()
    duration = round(end_time - start_time)
    print(f'Solution found for cluster {cluster} in {duration} sec')


Optimizing cluster 0...
Solution found for cluster 0 in 7 sec
Optimizing cluster 1...
Solution found for cluster 1 in 61 sec
Optimizing cluster 2...
Solution found for cluster 2 in 61 sec
Optimizing cluster 3...
Solution found for cluster 3 in 3 sec
Optimizing cluster 4...
Solution found for cluster 4 in 3 sec
Optimizing cluster 5...
Solution found for cluster 5 in 8 sec
Optimizing cluster 6...
Solution found for cluster 6 in 3 sec
Optimizing cluster 7...
Solution found for cluster 7 in 12 sec


In [321]:
print("\n===== TRUCK ROUTE REPORT (ALL CLUSTERS) =====\n")

global_truck_id = 0

for cluster, sol in solutions_dict.items():
    
    I = sol["I"]
    I_all = sol["I_all"]
    K = sol["K"]
    warehouse_id = sol["warehouse_id"]
    D = sol["D"]
    
    X = sol["X"]
    Y = sol["Y"]
    T = sol["T"]
    W = sol["W"]
    
    for k in K:
        
        print(f"\nTruck {global_truck_id} (cluster {cluster}) :")
        
        current = warehouse_id
        visited = set()
        total_km = 0
        
        while True:
            
            next_station = None
            
            for j in I_all:
                if W[current, j, k] > 0.5:
                    next_station = j
                    break
            
            if next_station is None:
                print("  No route.")
                break
            
            total_km += D[current][next_station]
            
            if next_station == warehouse_id:
                print(f"  Goes from {current} -> Warehouse (return) | "
                      f"Distance: {D[current][next_station]:.2f} km")
                break
            
            drop = X.get((next_station, k), 0)
            pick = Y.get((next_station, k), 0)
            load = T[(next_station, k)]
            
            print(f"  Goes from {current} -> {next_station} | "
                  f"Distance: {D[current][next_station]:.2f} km | "
                  f"Drop: {drop:.0f}, Pick: {pick:.0f}, "
                  f"Truck load before arrival: {load:.0f}")
            
            if next_station in visited:
                print("  WARNING: loop detected")
                break
            
            visited.add(next_station)
            current = next_station
        
        print(f"  --> Total distance traveled by Truck {global_truck_id}: {total_km:.2f} km")
        
        global_truck_id += 1


===== TRUCK ROUTE REPORT (ALL CLUSTERS) =====


Truck 0 (cluster 0) :
  Goes from 0 -> 214 | Distance: 0.33 km | Drop: 0, Pick: 8, Truck load before arrival: 0
  Goes from 214 -> 215 | Distance: 1.11 km | Drop: 3, Pick: 0, Truck load before arrival: 8
  Goes from 215 -> 146 | Distance: 1.14 km | Drop: 1, Pick: 0, Truck load before arrival: 5
  Goes from 146 -> 134 | Distance: 1.02 km | Drop: 4, Pick: 0, Truck load before arrival: 5
  Goes from 134 -> 88 | Distance: 0.92 km | Drop: 0, Pick: 12, Truck load before arrival: 1
  Goes from 88 -> 54 | Distance: 1.39 km | Drop: 3, Pick: 0, Truck load before arrival: 13
  Goes from 54 -> 111 | Distance: 1.31 km | Drop: 0, Pick: 15, Truck load before arrival: 10
  Goes from 111 -> 291 | Distance: 1.37 km | Drop: 3, Pick: 0, Truck load before arrival: 25
  Goes from 291 -> 268 | Distance: 0.87 km | Drop: 1, Pick: 0, Truck load before arrival: 23
  Goes from 268 -> 143 | Distance: 1.53 km | Drop: 3, Pick: 0, Truck load before arrival: 22
  Goes f

In [ ]:
center_lat = stations['latitude'].mean()
center_lon = stations['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

truck_colors = [
    'blue','red','green','orange','purple',
    'cyan','magenta','brown','pink','black'
]

global_truck_id = 0
offset_step = 0.00005

for cluster, sol in solutions_dict.items():
    
    I = sol["I"]
    I_all = sol["I_all"]
    K = sol["K"]
    warehouse_id = sol["warehouse_id"]
    
    X = sol["X"]
    Y = sol["Y"]
    W = sol["W"]
    
    for k in K:
        
        color = truck_colors[global_truck_id % len(truck_colors)]
        
        # --- reconstruct route ---
        route = [warehouse_id]
        visited = set()
        current = warehouse_id
        
        while True:
            
            next_stations = [j for j in I_all if W[current, j, k] > 0.5]
            
            if len(next_stations) == 0:
                break
            
            next_station = next_stations[0]
            
            if next_station == warehouse_id:
                route.append(warehouse_id)
                break
            
            if next_station in visited:
                break
            
            route.append(next_station)
            visited.add(next_station)
            current = next_station
        
        # --- draw route ---
        coords = [
            (
                stations.loc[stations['id']==s, 'latitude'].values[0],
                stations.loc[stations['id']==s, 'longitude'].values[0]
            )
            for s in route
        ]
        
        folium.PolyLine(
            coords,
            color=color,
            weight=4,
            opacity=0.7,
            tooltip=f"Truck {global_truck_id} (cluster {cluster})"
        ).add_to(m)
        
        # --- draw station actions ---
        for s in route:
            
            if s == warehouse_id:
                continue
            
            lat = stations.loc[stations['id']==s,'latitude'].values[0]
            lon = stations.loc[stations['id']==s,'longitude'].values[0]
            
            lat_offset = lat + global_truck_id * offset_step
            lon_offset = lon + global_truck_id * offset_step
            
            drop = X.get((s,k),0)
            pick = Y.get((s,k),0)
            
            tooltip = f"Truck {global_truck_id}: "
            
            if drop > 0:
                tooltip += f"Drop {drop:.0f}"
                
                folium.CircleMarker(
                    location=[lat_offset, lon_offset],
                    radius=5,
                    color='green',
                    fill=True,
                    fill_color='green',
                    fill_opacity=0.7,
                    tooltip=tooltip
                ).add_to(m)
            
            if pick > 0:
                tooltip += f" Pick {pick:.0f}"
                
                folium.CircleMarker(
                    location=[lat_offset, lon_offset],
                    radius=5,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.7,
                    tooltip=tooltip
                ).add_to(m)
        
        global_truck_id += 1

# --- Save and open map ---
m.save(f"../plots/bike_trucks_map_{hour_to_optimize[:-6]}.html")
#webbrowser.open(f"../plots/bike_trucks_map_{hour_to_optimize[:-6]}.html")

True